In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
pip install -U sentence-transformers transformers huggingface_hub tokenizers

In [2]:
train = pd.read_csv('train_processed.csv')

In [3]:
train.isnull().sum()

sample_id      0
value          0
model_input    0
log_price      0
image_name     0
dtype: int64

In [4]:
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

/home/gagan/Desktop/side-projects/amazon-mlchallenge/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from sklearn.model_selection import train_test_split
X = train[['value', 'model_input']]
y = train['log_price']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
import re
import string

def clean_text(text):
    if pd.isnull(text):
        return ""
    # Remove emojis
    text = re.sub(r'[\U00010000-\U0010ffff]', '', text)
    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Lowercase
    text = text.lower()
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Clean train/val/test model_input
X_train['model_input'] = X_train['model_input'].apply(clean_text)
X_val['model_input'] = X_val['model_input'].apply(clean_text)


In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the model
st_model = SentenceTransformer('msmarco-distilbert-base-v4')

def get_st_embeddings(texts, batch_size=32):
    # Returns a numpy array of embeddings
    return st_model.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)

# Example usage:
X_train_emb = get_st_embeddings(X_train['model_input'].tolist())
X_val_emb = get_st_embeddings(X_val['model_input'].tolist())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/545 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1852 [00:00<?, ?it/s]

Batches:   0%|          | 0/463 [00:00<?, ?it/s]

In [9]:
scaler = StandardScaler()
X_train_value = scaler.fit_transform(X_train[['value']])
X_val_value = scaler.transform(X_val[['value']])

In [10]:
X_train_combined = np.hstack([X_train_value, X_train_emb])
X_val_combined = np.hstack([X_val_value, X_val_emb])

In [11]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import numpy as np

class PriceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(np.array(y), dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = PriceDataset(X_train_combined, y_train.values)
val_ds = PriceDataset(X_val_combined, y_val.values)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=128)

# Improved Neural Network for dual encoder
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP(X_train_combined.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.SmoothL1Loss(beta=0.5)

# Training loop with early stopping
best_loss = float('inf')
patience = 5
counter = 0
epochs = 50

for epoch in range(epochs):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        loss = loss_fn(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    # Validation
    model.eval()
    val_losses = []
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            val_loss = loss_fn(pred, yb)
            val_losses.append(val_loss.item())
    avg_val_loss = np.mean(val_losses)
    print(f"Epoch {epoch+1}, Val Loss: {avg_val_loss:.4f}")
    # Early stopping
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping!")
            break

# Load best model and evaluate SMAPE
model.load_state_dict(torch.load("best_model.pt"))
model.eval()
preds = []
with torch.no_grad():
    for xb, _ in val_dl:
        xb = xb.to(device)
        pred = model(xb).cpu().numpy()
        preds.append(pred)
y_pred = np.concatenate(preds)

def smape(y_true, y_pred):
    y_true = np.expm1(y_true)
    y_pred = np.expm1(y_pred)
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))

print(f"Validation SMAPE: {smape(y_val.values, y_pred):.2f}%")

Epoch 1, Val Loss: 0.4129
Epoch 2, Val Loss: 0.3909
Epoch 3, Val Loss: 0.3866
Epoch 4, Val Loss: 0.3751
Epoch 5, Val Loss: 0.3711
Epoch 6, Val Loss: 0.3641
Epoch 7, Val Loss: 0.3673
Epoch 8, Val Loss: 0.3584
Epoch 9, Val Loss: 0.3584
Epoch 10, Val Loss: 0.3597
Epoch 11, Val Loss: 0.3599
Epoch 12, Val Loss: 0.3557
Epoch 13, Val Loss: 0.3622
Epoch 14, Val Loss: 0.3522
Epoch 15, Val Loss: 0.3529
Epoch 16, Val Loss: 0.3609
Epoch 17, Val Loss: 0.4374
Epoch 18, Val Loss: 0.3550
Epoch 19, Val Loss: 0.5281
Early stopping!
Validation SMAPE: 54.47%


In [13]:
from catboost import CatBoostRegressor, Pool

# Use y_train.values and y_val.values directly (already log1p(price))
cb_train = Pool(
    data=X_train[['value', 'model_input']],
    label=y_train.values,
    text_features=[1]
)
cb_val = Pool(
    data=X_val[['value', 'model_input']],
    label=y_val.values,
    text_features=[1]
)

cb_model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.03,
    depth=8,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=42,
    verbose=200,
    early_stopping_rounds=200,
    task_type='GPU' if torch.cuda.is_available() else 'CPU'
)
cb_model.fit(cb_train, eval_set=cb_val)
with torch.no_grad():
    nn_val_logits = []
    for xb, _ in val_dl:
        nn_val_logits.append(model(xb.to(device)).cpu().numpy())
nn_val_log = np.concatenate(nn_val_logits)
cb_val_log = cb_model.predict(cb_val)

# Optimize blend weight by validation SMAPE
def smape_log(y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))

weights = np.linspace(0.0, 1.0, 21)
best_w, best_s = 0.5, 1e9
for w in weights:
    blended = w * nn_val_log + (1 - w) * cb_val_log
    s = smape_log(y_val.values, blended)
    if s < best_s:
        best_s, best_w = s, w
print(f"Best blend w={best_w:.2f}, Val SMAPE={best_s:.2f}%")


0:	learn: 0.9338126	test: 0.9266773	best: 0.9266773 (0)	total: 7.58ms	remaining: 22.7s
200:	learn: 0.7567443	test: 0.7577910	best: 0.7577910 (200)	total: 1.05s	remaining: 14.6s
400:	learn: 0.7265550	test: 0.7343710	best: 0.7343710 (400)	total: 2.09s	remaining: 13.6s
600:	learn: 0.7068965	test: 0.7213966	best: 0.7213966 (600)	total: 3.12s	remaining: 12.5s
800:	learn: 0.6919633	test: 0.7132177	best: 0.7132177 (800)	total: 4.17s	remaining: 11.5s
1000:	learn: 0.6800685	test: 0.7073267	best: 0.7073267 (1000)	total: 5.22s	remaining: 10.4s
1200:	learn: 0.6697527	test: 0.7025972	best: 0.7025972 (1200)	total: 6.25s	remaining: 9.36s
1400:	learn: 0.6603566	test: 0.6988889	best: 0.6988889 (1400)	total: 7.27s	remaining: 8.29s
1600:	learn: 0.6517067	test: 0.6958424	best: 0.6958424 (1600)	total: 8.29s	remaining: 7.24s
1800:	learn: 0.6438584	test: 0.6932675	best: 0.6932675 (1800)	total: 9.31s	remaining: 6.2s
2000:	learn: 0.6364906	test: 0.6910171	best: 0.6910171 (2000)	total: 10.3s	remaining: 5.17s
22

In [14]:
test = pd.read_csv('/kaggle/input/amazon-test-processed/test_processed.csv')

In [15]:
test.isnull().sum()

sample_id      0
value          0
model_input    1
dtype: int64

In [16]:
test['model_input'] = test['model_input'].fillna('')

In [18]:
# Clean and embed test data
test['model_input'] = test['model_input'].apply(clean_text)
X_test_emb = get_st_embeddings(test['model_input'].tolist())
X_test_value = scaler.transform(test[['value']])
X_test_combined = np.hstack([X_test_value, X_test_emb])

# MLP predictions (log space)
test_ds = PriceDataset(X_test_combined, np.zeros(len(test)))  # dummy y
test_dl = DataLoader(test_ds, batch_size=128)
mlp_preds = []
model.eval()
with torch.no_grad():
    for xb, _ in test_dl:
        xb = xb.to(device)
        pred = model(xb).cpu().numpy()
        mlp_preds.append(pred)
mlp_test_log = np.concatenate(mlp_preds).flatten()

# CatBoost predictions (log space)
cb_test = Pool(
    data=test[['value', 'model_input']],
    text_features=[1]
)
cb_test_log = cb_model.predict(cb_test)

# Blend using best_w from validation
blended_test_log = best_w * mlp_test_log + (1 - best_w) * cb_test_log

# Convert back to price
blended_test_price = np.expm1(blended_test_log).clip(0)

# Prepare submission
submission = pd.DataFrame({
    'sample_id': test['sample_id'],
    'price': blended_test_price
})
submission.to_csv('submission.csv', index=False)
print("Submission file saved as submission.csv")

Batches:   0%|          | 0/2344 [00:00<?, ?it/s]

Submission file saved as submission.csv
